# Visualize GNN graph windows

Interactive exploration notebook for PyG `HeteroData` graph windows used by the GNN experiments.

This notebook compares the same window as:

- `GNN_v1`: `temporal` + `spatial` relations
- `GNN_v2`: `temporal_forward`, `temporal_backward`, `spatial`, and optional `fixation` relations

Node positions come from `x-avg`/`y-avg`. Node size uses robustly scaled mean pupil size: `(pupil-size-left-avg + pupil-size-right-avg) / 2`, clipped to the 5th and 95th percentile within the displayed nodes. Node labels show the local displayed node index, starting at 0.

By default this notebook first selects a small reproducible `(subject, recording)` subset and then builds shorter visualization windows (`VISUAL_WINDOW_LENGTH_SECONDS = 2.0`) with the same graph-construction rules. This keeps the notebook fast and more readable than loading the complete dataset or hiding arbitrary nodes from a dense 10-second training graph. Edge traces are curved by relation so overlapping multi-edge connections remain visible. For plotting only, bidirectional edges inside the same relation are collapsed to one visible line by default.

## Setup

Run this notebook from the repository root, or from the `notebooks/` directory. It uses the same dataset construction code as training, with graph-version overrides for side-by-side comparison.

In [1]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
import sys
from typing import Any, Mapping

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    go = None
    make_subplots = None

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from data.data import SpacioTemporalDataset
from emotions.common.dataset_config import (
    build_graph_dataset_kwargs,
    resolve_dropna_columns,
    resolve_feature_columns,
)

print(f"Repo root: {REPO_ROOT}")
print(f"Plotly available: {PLOTLY_AVAILABLE}")
if not PLOTLY_AVAILABLE:
    print("For zoom/pan/hover Plotly views, install it in the gfm environment, e.g. `python -m pip install plotly`.")

Repo root: /home/ppg/eyetracking/GFM-for-eyetracker-X-diploma/GFM-for-eyetracker
Plotly available: True


## Configuration

The defaults point to the active quick v1/v2 Table-6 wrapper. For graph visualization, the notebook uses the same dataset construction settings and optionally overrides only the window length for readability.

In [2]:
BASE_CONFIG_PATH = REPO_ROOT / "src/emotions/gnn_improvement_experiments/configs/quick_v1_v2/run_hci_experiment_suite_table6_3class.yaml"

# Dataset subset controls. Random selection is reproducible and happens before graph construction.
USE_RANDOM_SUBSET = True
SUBSET_RANDOM_SEED = 42
N_SUBJECT_RECORDING_PAIRS = 1
FILTER_SUBJECT = None       # example: "P1"; overrides random subject selection when set
FILTER_RECORDING = None     # example: "69.avi"; overrides random recording selection when set
SELECTED_GRAPH_INDEX = 0

# Readability controls. Short visualization windows avoid displaying arbitrary node subsets.
USE_SHORT_VISUAL_WINDOWS = True
VISUAL_WINDOW_LENGTH_SECONDS = 2.0
NODE_START = 0
MAX_NODES_TO_DRAW = None    # None means draw every node in the selected visualization window.
MAX_EDGES_PER_RELATION = None
COLLAPSE_BIDIRECTIONAL_EDGES_FOR_PLOT = True
INVERT_Y_AXIS = True
SHOW_CURVED_EDGES = True
EDGE_CURVE_POINTS = 13
SHOW_NODE_INDEX_LABELS = True
DISTANCE_LABEL_UNIT = "cm"
NODE_SIZE_MIN = 18.0
NODE_SIZE_MAX = 38.0
NODE_LABEL_FONT_SIZE = 9
EDGE_ALPHA = 0.50

RELATION_LABELS = {
    "temporal": "Temporal (v1)",
    "temporal_forward": "Temporal forward (v2)",
    "temporal_backward": "Temporal backward (v2)",
    "spatial": "Spatial kNN",
    "fixation": "Same fixation",
}

EDGE_STYLE = {
    "temporal": {"color": "#ff7f0e", "width": 3.0, "curve": 0.22},
    "temporal_forward": {"color": "#2ca02c", "width": 2.8, "curve": 0.18},
    "temporal_backward": {"color": "#d62728", "width": 2.8, "curve": 0.18},
    "spatial": {"color": "#1f77b4", "width": 2.4, "curve": 0.08},
    "fixation": {"color": "#9467bd", "width": 3.2, "curve": -0.30},
}

In [3]:
def deep_update(base: dict[str, Any], overrides: Mapping[str, Any]) -> dict[str, Any]:
    """Recursively merge mapping values into a copied configuration dictionary."""
    result = deepcopy(base)
    for key, value in overrides.items():
        if isinstance(value, Mapping) and isinstance(result.get(key), Mapping):
            result[key] = deep_update(dict(result[key]), value)
        else:
            result[key] = deepcopy(value)
    return result


def load_training_config(wrapper_config_path: Path) -> dict[str, Any]:
    """Load the quick v1/v2 wrapper and apply its global overrides to the multiclass base config."""
    with wrapper_config_path.open("r", encoding="utf-8") as handle:
        wrapper = yaml.safe_load(handle) or {}
    trainer_path = REPO_ROOT / wrapper["suite"]["base_configs"]["multiclass"]
    with trainer_path.open("r", encoding="utf-8") as handle:
        config = yaml.safe_load(handle) or {}
    return deep_update(config, wrapper.get("global_overrides", {}))


def resolve_data_path(dataset_cfg: Mapping[str, Any]) -> Path:
    """Return the configured single-file dataset path as an absolute path."""
    data_filepath = dataset_cfg.get("data_filepath")
    if data_filepath is None:
        raise ValueError("This notebook expects dataset.data_filepath to point to the merged HCI CSV.")
    return REPO_ROOT / str(data_filepath)


def choose_subject_recording_subset(config: dict[str, Any]) -> pd.DataFrame:
    """Choose a small subject-recording subset before constructing graph windows."""
    dataset_cfg = config["dataset"]
    data_path = resolve_data_path(dataset_cfg)
    columns = [
        "subject",
        "recording",
        str(config["multiclass_task"].get("target_column", "emotion-id")),
    ]
    optional_columns = [
        dataset_cfg.get("experiment_type_column", "experiment-type"),
        dataset_cfg.get("label_quality_column"),
    ]
    usecols = [column for column in dict.fromkeys(columns + optional_columns) if column is not None]
    df = pd.read_csv(data_path, usecols=lambda column: column in set(usecols))

    experiment_type_column = dataset_cfg.get("experiment_type_column", "experiment-type")
    allowed_experiment_types = dataset_cfg.get("allowed_experiment_types")
    if allowed_experiment_types and experiment_type_column in df.columns:
        df = df[df[experiment_type_column].isin(allowed_experiment_types)]

    label_quality_column = dataset_cfg.get("label_quality_column")
    allowed_label_quality_values = dataset_cfg.get("allowed_label_quality_values")
    if label_quality_column and allowed_label_quality_values and label_quality_column in df.columns:
        df = df[df[label_quality_column].isin(allowed_label_quality_values)]

    exclude_subjects = dataset_cfg.get("exclude_subjects")
    if exclude_subjects:
        df = df[~df["subject"].isin(exclude_subjects)]

    if FILTER_SUBJECT is not None:
        df = df[df["subject"].astype(str) == str(FILTER_SUBJECT)]
    if FILTER_RECORDING is not None:
        df = df[df["recording"].astype(str) == str(FILTER_RECORDING)]
    if len(df) == 0:
        raise ValueError("No rows remain after applying visualization subset filters.")

    pairs = df[["subject", "recording"]].drop_duplicates().sort_values(["subject", "recording"]).reset_index(drop=True)
    if USE_RANDOM_SUBSET and (FILTER_SUBJECT is None or FILTER_RECORDING is None):
        n_pairs = min(int(N_SUBJECT_RECORDING_PAIRS), len(pairs))
        pairs = pairs.sample(n=n_pairs, random_state=int(SUBSET_RANDOM_SEED)).sort_values(["subject", "recording"]).reset_index(drop=True)
    else:
        pairs = pairs.head(int(N_SUBJECT_RECORDING_PAIRS)).reset_index(drop=True)
    return pairs


def apply_visualization_subset(config: dict[str, Any], pairs: pd.DataFrame) -> dict[str, Any]:
    """Apply a small subject-recording subset to the dataset config."""
    subset_config = deepcopy(config)
    dataset_cfg = subset_config["dataset"]
    dataset_cfg["filter_subjects"] = sorted(pairs["subject"].astype(str).unique().tolist())
    dataset_cfg["filter_recordings"] = sorted(pairs["recording"].astype(str).unique().tolist())
    return subset_config


def graph_dataset_kwargs_for_version(config: dict[str, Any], graph_version: str) -> dict[str, Any]:
    """Build dataset kwargs matching training, with visualization-only overrides."""
    if graph_version not in {"v1", "v2"}:
        raise ValueError("graph_version must be 'v1' or 'v2'.")

    dataset_cfg = deepcopy(config["dataset"])
    dataset_cfg["graph_version"] = graph_version
    dataset_cfg["edge_weight_mode"] = "handcrafted" if graph_version == "v1" else dataset_cfg.get("edge_weight_mode", "learned_signed")
    if USE_SHORT_VISUAL_WINDOWS:
        dataset_cfg["window_length"] = float(VISUAL_WINDOW_LENGTH_SECONDS)
        kt = int(dataset_cfg.get("kt", 1))
        ks = int(dataset_cfg.get("ks", 1))
        original_min_samples = int(dataset_cfg.get("min_samples_per_window", max(kt, ks) + 1))
        visual_min_samples = max(max(kt, ks) + 1, int(round(float(VISUAL_WINDOW_LENGTH_SECONDS) * 30)))
        dataset_cfg["min_samples_per_window"] = min(original_min_samples, visual_min_samples)

    target_column = str(config["multiclass_task"].get("target_column", "emotion-id"))
    feature_columns = resolve_feature_columns(dataset_cfg)
    dropna_columns = resolve_dropna_columns(dataset_cfg, target_columns=[target_column])

    kwargs = build_graph_dataset_kwargs(
        dataset_cfg=dataset_cfg,
        target_columns=[target_column],
        feature_columns=feature_columns,
        dropna_columns=dropna_columns,
    )
    if kwargs["data_filepath"] is not None:
        kwargs["data_filepath"] = str(REPO_ROOT / kwargs["data_filepath"])
    if kwargs["root_dir"] is not None:
        kwargs["root_dir"] = str(REPO_ROOT / kwargs["root_dir"])
    if kwargs["cache_dir"] is not None:
        kwargs["cache_dir"] = str(REPO_ROOT / kwargs["cache_dir"])
    return kwargs


def build_dataset_pair(config: dict[str, Any]) -> tuple[SpacioTemporalDataset, SpacioTemporalDataset]:
    """Construct matching v1 and v2 graph datasets for side-by-side window comparison."""
    v1 = SpacioTemporalDataset(**graph_dataset_kwargs_for_version(config, "v1"))
    v2 = SpacioTemporalDataset(**graph_dataset_kwargs_for_version(config, "v2"))
    if len(v1) != len(v2):
        raise ValueError(f"v1 and v2 dataset lengths differ: {len(v1)} vs {len(v2)}")
    return v1, v2


def graph_metadata(dataset: SpacioTemporalDataset) -> pd.DataFrame:
    """Summarize available graph windows for easy selection."""
    rows = []
    for idx, graph in enumerate(dataset.graphs):
        y = getattr(graph, "y", None)
        target = None if y is None else np.asarray(y).reshape(-1).tolist()
        rows.append({
            "graph_index": idx,
            "subject": getattr(graph, "subject", None),
            "recording": getattr(graph, "recording", None),
            "source_file": getattr(graph, "source_file", None),
            "num_nodes": int(graph["node"].num_nodes),
            "target": target,
        })
    return pd.DataFrame(rows)


def choose_graph_index(metadata: pd.DataFrame, *, subject: str | None, recording: str | None, fallback_index: int) -> int:
    """Choose the first matching graph index, or return the configured fallback index."""
    selected = metadata
    if subject is not None:
        selected = selected[selected["subject"].astype(str) == str(subject)]
    if recording is not None:
        selected = selected[selected["recording"].astype(str) == str(recording)]
    if len(selected) > 0:
        return int(selected.iloc[0]["graph_index"])
    return int(fallback_index)

## Build matching v1/v2 datasets

This cell first chooses a small reproducible subject-recording subset from the merged HCI CSV, then constructs matching v1/v2 graph windows only for that subset. This keeps notebook startup fast and avoids loading the complete dataset for visualization.

In [4]:
config_full = load_training_config(BASE_CONFIG_PATH)
selected_pairs = choose_subject_recording_subset(config_full)
config = apply_visualization_subset(config_full, selected_pairs)

display(selected_pairs.assign(selection="visualization subset"))

dataset_v1, dataset_v2 = build_dataset_pair(config)
metadata = graph_metadata(dataset_v2)

window_label = f"{VISUAL_WINDOW_LENGTH_SECONDS:g}s visualization" if USE_SHORT_VISUAL_WINDOWS else f"{config['dataset']['window_length']}s training"
print(f"Loaded {len(dataset_v2):,} matching {window_label} graph windows from {len(selected_pairs)} subject-recording pair(s).")
display(metadata.head(10))
metadata[["num_nodes"]].describe().T

,subject,recording,selection
0,P5,newyork_f.avi,visualization subset


Loaded data from /home/ppg/eyetracking/GFM-for-eyetracker-X-diploma/GFM-for-eyetracker/data/processed/cached_hci_tagging_emotion.csv: 1 subject-recording pairs
Loaded 68 graphs from /home/ppg/eyetracking/GFM-for-eyetracker-X-diploma/GFM-for-eyetracker/data/processed/cached_hci_tagging_emotion.csv
Saving dataset to cache: /home/ppg/eyetracking/GFM-for-eyetracker-X-diploma/GFM-for-eyetracker/data/cache/dataset_kt2_ks2_tau0.05_wl2.0_wo0_9b13d5da.pkl
Successfully cached 68 graphs
Loaded data from /home/ppg/eyetracking/GFM-for-eyetracker-X-diploma/GFM-for-eyetracker/data/processed/cached_hci_tagging_emotion.csv: 1 subject-recording pairs
Loaded 68 graphs from /home/ppg/eyetracking/GFM-for-eyetracker-X-diploma/GFM-for-eyetracker/data/processed/cached_hci_tagging_emotion.csv
Saving dataset to cache: /home/ppg/eyetracking/GFM-for-eyetracker-X-diploma/GFM-for-eyetracker/data/cache/dataset_kt2_ks2_tau0.05_wl2.0_wo0_6d67cc7f.pkl
Successfully cached 68 graphs
Loaded 68 matching 2s visualization gr

,graph_index,subject,recording,source_file,num_nodes,target
0,0,P5,newyork_f.avi,subject_P5_recording_newyork_f.avi.csv,123,[6.0]
1,1,P5,newyork_f.avi,subject_P5_recording_newyork_f.avi.csv,121,[6.0]
2,2,P5,newyork_f.avi,subject_P5_recording_newyork_f.avi.csv,121,[6.0]
3,3,P5,newyork_f.avi,subject_P5_recording_newyork_f.avi.csv,119,[6.0]
4,4,P5,newyork_f.avi,subject_P5_recording_newyork_f.avi.csv,121,[6.0]
5,5,P5,newyork_f.avi,subject_P5_recording_newyork_f.avi.csv,121,[6.0]
6,6,P5,newyork_f.avi,subject_P5_recording_newyork_f.avi.csv,121,[6.0]
7,7,P5,newyork_f.avi,subject_P5_recording_newyork_f.avi.csv,120,[6.0]
8,8,P5,newyork_f.avi,subject_P5_recording_newyork_f.avi.csv,121,[6.0]
9,9,P5,newyork_f.avi,subject_P5_recording_newyork_f.avi.csv,121,[6.0]


,count,mean,std,min,25%,50%,75%,max
num_nodes,68.0,117.632353,9.758126,81.0,120.0,121.0,121.0,123.0


## Optional selector

Use the widget to pick a graph index interactively within the loaded visualization subset. The default view draws all nodes in the selected shorter visualization window, so the plot is not a random node subset.

In [5]:
default_graph_index = choose_graph_index(
    metadata,
    subject=FILTER_SUBJECT,
    recording=FILTER_RECORDING,
    fallback_index=SELECTED_GRAPH_INDEX,
)

if widgets is not None:
    option_rows = metadata.head(500).copy()
    option_rows["label"] = option_rows.apply(
        lambda row: f"{int(row.graph_index)} | {row.subject} | {row.recording} | n={int(row.num_nodes)}",
        axis=1,
    )
    graph_index_widget = widgets.Dropdown(
        options=list(zip(option_rows["label"], option_rows["graph_index"].astype(int))),
        value=default_graph_index if default_graph_index in set(option_rows["graph_index"].astype(int)) else int(option_rows.iloc[0]["graph_index"]),
        description="Graph",
        layout=widgets.Layout(width="650px"),
    )
    node_start_widget = widgets.IntText(value=NODE_START, description="Node start")
    max_nodes_widget = widgets.IntText(value=-1 if MAX_NODES_TO_DRAW is None else int(MAX_NODES_TO_DRAW), description="Max nodes")
    display(widgets.VBox([graph_index_widget, widgets.HBox([node_start_widget, max_nodes_widget])]))
else:
    graph_index_widget = None
    node_start_widget = None
    max_nodes_widget = None
    print(f"Using graph index: {default_graph_index}")

## Plot helpers

The Plotly figure supports zoom, pan, hover text, and legend toggles. The Matplotlib fallback keeps the same geometry and relation colors.

In [6]:
def tensor_to_numpy(value: torch.Tensor | np.ndarray) -> np.ndarray:
    """Move a tensor-like value to a NumPy array."""
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def feature_index(feature_columns: list[str], name: str) -> int:
    """Return the index for a named node feature."""
    if name not in feature_columns:
        raise ValueError(f"Feature {name!r} is missing from feature_columns={feature_columns}")
    return feature_columns.index(name)


def displayed_node_indices(num_nodes: int, start: int, max_nodes: int | None) -> np.ndarray:
    """Return the local node indices that should be drawn."""
    start = max(0, int(start))
    if max_nodes is None or int(max_nodes) < 0:
        stop = num_nodes
    else:
        stop = min(num_nodes, start + int(max_nodes))
    return np.arange(start, stop, dtype=int)


def robust_marker_sizes(values: np.ndarray, *, min_size: float = NODE_SIZE_MIN, max_size: float = NODE_SIZE_MAX) -> np.ndarray:
    """Scale values to marker diameters after 5th/95th percentile clipping."""
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return np.full(values.shape, (min_size + max_size) / 2.0)
    low, high = np.nanpercentile(finite, [5, 95])
    if not np.isfinite(low) or not np.isfinite(high) or np.isclose(low, high):
        return np.full(values.shape, (min_size + max_size) / 2.0)
    clipped = np.clip(values, low, high)
    return min_size + (clipped - low) / (high - low) * (max_size - min_size)


def distance_labels(distance_values: np.ndarray, unit: str = DISTANCE_LABEL_UNIT) -> tuple[np.ndarray, list[str]]:
    """Convert distance values to compact node labels."""
    values = np.asarray(distance_values, dtype=float)
    if unit != "cm":
        raise ValueError(f"Unsupported DISTANCE_LABEL_UNIT={unit!r}. Currently supported: 'cm'.")
    converted = values / 10.0  # Tobii distance columns are millimeter-scale; labels use centimeters.
    labels = ["" if not np.isfinite(value) else f"{int(round(value))}" for value in converted]
    return converted, labels


def graph_node_frame(graph: Any, feature_columns: list[str], node_indices: np.ndarray) -> pd.DataFrame:
    """Extract display-ready node attributes from one graph."""
    x_matrix = tensor_to_numpy(graph["node"].x)
    x_idx = feature_index(feature_columns, "x-avg")
    y_idx = feature_index(feature_columns, "y-avg")
    left_idx = feature_index(feature_columns, "pupil-size-left-avg")
    right_idx = feature_index(feature_columns, "pupil-size-right-avg")
    distance_idx = feature_index(feature_columns, "distance-avg")

    display_order = np.arange(len(node_indices))
    pupil_mean = np.nanmean(x_matrix[:, [left_idx, right_idx]], axis=1)
    distance_raw = x_matrix[:, distance_idx]
    distance_cm, _ = distance_labels(distance_raw[node_indices])
    sizes = robust_marker_sizes(pupil_mean[node_indices])
    return pd.DataFrame({
        "node": node_indices,
        "x": x_matrix[node_indices, x_idx],
        "y": x_matrix[node_indices, y_idx],
        "pupil_mean": pupil_mean[node_indices],
        "distance_raw": distance_raw[node_indices],
        "distance_cm": distance_cm,
        "node_label": [str(int(index)) for index in display_order],
        "marker_size": sizes,
        "display_order": display_order,
    })


def relation_edge_index(graph: Any, relation: str) -> np.ndarray:
    """Return an edge index array for one relation, or an empty array."""
    edge_type = ("node", relation, "node")
    if edge_type not in graph.edge_types:
        return np.empty((2, 0), dtype=int)
    return tensor_to_numpy(graph[edge_type].edge_index).astype(int)


def filtered_edges(edge_index: np.ndarray, node_indices: np.ndarray, max_edges: int | None) -> np.ndarray:
    """Keep directed edges whose endpoints are both visible, with optional deterministic thinning."""
    if edge_index.size == 0:
        return np.empty((2, 0), dtype=int)
    visible = set(int(i) for i in node_indices)
    keep_mask = np.array([(int(src) in visible and int(dst) in visible) for src, dst in edge_index.T], dtype=bool)
    edges = edge_index[:, keep_mask]
    if max_edges is not None and edges.shape[1] > int(max_edges):
        selected = np.linspace(0, edges.shape[1] - 1, int(max_edges)).round().astype(int)
        edges = edges[:, selected]
    return edges


def collapse_bidirectional_edges(edge_index: np.ndarray) -> np.ndarray:
    """Collapse exact and reverse duplicates to one displayed undirected line per node pair."""
    if edge_index.size == 0:
        return np.empty((2, 0), dtype=int)
    seen: set[tuple[int, int]] = set()
    kept_edges: list[tuple[int, int]] = []
    for src, dst in edge_index.T:
        src_int = int(src)
        dst_int = int(dst)
        key = (min(src_int, dst_int), max(src_int, dst_int))
        if key in seen:
            continue
        seen.add(key)
        kept_edges.append((src_int, dst_int))
    if not kept_edges:
        return np.empty((2, 0), dtype=int)
    return np.asarray(kept_edges, dtype=int).T


def display_edges(edge_index: np.ndarray, node_indices: np.ndarray, max_edges: int | None) -> np.ndarray:
    """Return edges used for plotting after visibility filtering and optional bidirectional collapse."""
    edges = filtered_edges(edge_index, node_indices, max_edges=max_edges)
    if COLLAPSE_BIDIRECTIONAL_EDGES_FOR_PLOT:
        edges = collapse_bidirectional_edges(edges)
    return edges


def relation_order(graph: Any) -> list[str]:
    """Return known relations in a stable visual order."""
    candidates = ["spatial", "temporal", "temporal_forward", "temporal_backward", "fixation"]
    return [rel for rel in candidates if ("node", rel, "node") in graph.edge_types]


def curved_edge_coordinates(
    edges: np.ndarray,
    node_positions: pd.DataFrame,
    relation: str,
    *,
    show_curves: bool,
    curve_points: int = EDGE_CURVE_POINTS,
) -> tuple[list[float], list[float]]:
    """Build segmented edge coordinates, curving relation types apart for multi-edge readability."""
    if edges.shape[1] == 0:
        return [], []

    pos = node_positions.set_index("node")[["x", "y"]]
    relation_curve = EDGE_STYLE.get(relation, {}).get("curve", 0.0) if show_curves else 0.0
    t_values = np.linspace(0.0, 1.0, max(int(curve_points), 2))

    xs: list[float] = []
    ys: list[float] = []
    for src, dst in edges.T:
        x0, y0 = pos.loc[int(src)]
        x1, y1 = pos.loc[int(dst)]
        dx = float(x1 - x0)
        dy = float(y1 - y0)
        length = max((dx * dx + dy * dy) ** 0.5, 1e-9)
        normal_x = -dy / length
        normal_y = dx / length
        mid_x = float((x0 + x1) / 2.0)
        mid_y = float((y0 + y1) / 2.0)
        control_x = mid_x + relation_curve * length * normal_x
        control_y = mid_y + relation_curve * length * normal_y

        curve_x = (1 - t_values) ** 2 * float(x0) + 2 * (1 - t_values) * t_values * control_x + t_values ** 2 * float(x1)
        curve_y = (1 - t_values) ** 2 * float(y0) + 2 * (1 - t_values) * t_values * control_y + t_values ** 2 * float(y1)
        xs.extend(curve_x.astype(float).tolist() + [None])
        ys.extend(curve_y.astype(float).tolist() + [None])
    return xs, ys


def edge_count_table(graphs: dict[str, Any], node_indices: np.ndarray, max_edges: int | None) -> pd.DataFrame:
    """Summarize true directed edge counts and plotting edge counts by relation."""
    rows = []
    for graph_name, graph in graphs.items():
        for relation in relation_order(graph):
            edge_index = relation_edge_index(graph, relation)
            visible_directed_edges = filtered_edges(edge_index, node_indices, max_edges=max_edges)
            plotted_edges = display_edges(edge_index, node_indices, max_edges=max_edges)
            rows.append({
                "graph": graph_name,
                "relation": relation,
                "total_directed_edges": int(edge_index.shape[1]),
                "visible_directed_edges": int(visible_directed_edges.shape[1]),
                "displayed_edges": int(plotted_edges.shape[1]),
                "collapsed_for_plot": int(visible_directed_edges.shape[1] - plotted_edges.shape[1]),
            })
    return pd.DataFrame(rows)

In [7]:
def plot_graph_pair_plotly(
    graph_v1: Any,
    graph_v2: Any,
    *,
    feature_columns: list[str],
    node_start: int,
    max_nodes: int | None,
    max_edges_per_relation: int | None,
    invert_y_axis: bool,
    show_curved_edges: bool,
    show_node_index_labels: bool = SHOW_NODE_INDEX_LABELS,
):
    """Create an interactive Plotly side-by-side visualization for v1 and v2 graphs."""
    if not PLOTLY_AVAILABLE:
        raise ImportError("Plotly is not installed in this environment.")

    node_indices = displayed_node_indices(graph_v2["node"].num_nodes, node_start, max_nodes)
    graphs = {"GNN v1": graph_v1, "GNN v2": graph_v2}
    fig = make_subplots(rows=1, cols=2, subplot_titles=list(graphs.keys()), horizontal_spacing=0.06)
    legend_seen: set[str] = set()

    for col, (graph_name, graph) in enumerate(graphs.items(), start=1):
        node_frame = graph_node_frame(graph, feature_columns, node_indices)
        for relation in relation_order(graph):
            edge_index = relation_edge_index(graph, relation)
            edges = display_edges(edge_index, node_indices, max_edges=max_edges_per_relation)
            xs, ys = curved_edge_coordinates(edges, node_frame, relation, show_curves=show_curved_edges)
            style = EDGE_STYLE.get(relation, {"color": "#666666", "width": 2.0})
            fig.add_trace(
                go.Scatter(
                    x=xs,
                    y=ys,
                    mode="lines",
                    line={"color": style["color"], "width": style["width"]},
                    opacity=EDGE_ALPHA,
                    name=RELATION_LABELS.get(relation, relation),
                    legendgroup=relation,
                    showlegend=(relation not in legend_seen),
                    hoverinfo="skip",
                ),
                row=1,
                col=col,
            )
            legend_seen.add(relation)

        hover = [
            f"node={int(row.node)}<br>x={row.x:.3f}<br>y={row.y:.3f}<br>mean pupil={row.pupil_mean:.3f}<br>distance={row.distance_cm:.0f} cm ({row.distance_raw:.1f} raw)"
            for row in node_frame.itertuples(index=False)
        ]
        fig.add_trace(
            go.Scatter(
                x=node_frame["x"],
                y=node_frame["y"],
                mode="markers+text" if show_node_index_labels else "markers",
                text=node_frame["node_label"] if show_node_index_labels else None,
                textposition="middle center",
                textfont={"size": NODE_LABEL_FONT_SIZE, "color": "#111111"},
                marker={
                    "size": node_frame["marker_size"],
                    "color": node_frame["display_order"],
                    "colorscale": "Blues",
                    "line": {"color": "white", "width": 1.0},
                    "opacity": 0.80,
                    "showscale": (col == 2),
                    "colorbar": {
                        "title": {"text": "Node color: time order", "side": "bottom"},
                        "orientation": "h",
                        "thickness": 12,
                        "len": 0.42,
                        "x": 0.50,
                        "xanchor": "center",
                        "y": -0.18,
                        "yanchor": "top",
                    },
                },
                customdata=hover,
                hovertemplate="%{customdata}<extra></extra>",
                name=f"{graph_name} nodes: color=time order, size=pupil, label=node index",
                showlegend=(col == 1),
                legendgroup="nodes",
            ),
            row=1,
            col=col,
        )

    title = (
        f"Graph window {getattr(graph_v2, 'idx', [''])[0] if hasattr(graph_v2, 'idx') else ''} "
        f"| subject={getattr(graph_v2, 'subject', None)} "
        f"| recording={getattr(graph_v2, 'recording', None)} "
        f"| displayed nodes={len(node_indices)}/{graph_v2['node'].num_nodes} "
        f"| node color=time order, size=pupil, label=node index"
    )
    fig.update_layout(
        title=title,
        width=1400,
        height=740,
        template="plotly_white",
        hovermode="closest",
        legend={
            "title": {"text": "Edges and nodes"},
            "orientation": "v",
            "x": 1.02,
            "y": 1.0,
            "xanchor": "left",
            "yanchor": "top",
            "itemsizing": "constant",
        },
        margin={"l": 55, "r": 270, "t": 85, "b": 120},
    )
    fig.add_annotation(
        text="Edges: legend colors. Nodes: color=time order, size=mean pupil, label=local node index.",
        xref="paper",
        yref="paper",
        x=0.5,
        y=-0.12,
        showarrow=False,
        font={"size": 12, "color": "#333333"},
    )
    for col in [1, 2]:
        fig.update_xaxes(title_text="x-avg", row=1, col=col)
        fig.update_yaxes(title_text="y-avg", row=1, col=col, autorange="reversed" if invert_y_axis else True)
    return fig


def plot_graph_pair_matplotlib(
    graph_v1: Any,
    graph_v2: Any,
    *,
    feature_columns: list[str],
    node_start: int,
    max_nodes: int | None,
    max_edges_per_relation: int | None,
    invert_y_axis: bool,
    show_curved_edges: bool,
    show_node_index_labels: bool = SHOW_NODE_INDEX_LABELS,
):
    """Create a static Matplotlib fallback with the same geometry and relation colors."""
    node_indices = displayed_node_indices(graph_v2["node"].num_nodes, node_start, max_nodes)
    graphs = {"GNN v1": graph_v1, "GNN v2": graph_v2}
    fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
    relation_handles = {}

    for ax, (graph_name, graph) in zip(axes, graphs.items()):
        node_frame = graph_node_frame(graph, feature_columns, node_indices)
        for relation in relation_order(graph):
            edge_index = relation_edge_index(graph, relation)
            edges = display_edges(edge_index, node_indices, max_edges=max_edges_per_relation)
            xs, ys = curved_edge_coordinates(edges, node_frame, relation, show_curves=show_curved_edges)
            style = EDGE_STYLE.get(relation, {"color": "#666666", "width": 2.0})
            line, = ax.plot(
                xs,
                ys,
                color=style["color"],
                linewidth=style["width"],
                alpha=EDGE_ALPHA,
                label=RELATION_LABELS.get(relation, relation),
                zorder=2,
            )
            relation_handles.setdefault(relation, line)

        scatter = ax.scatter(
            node_frame["x"],
            node_frame["y"],
            s=np.square(node_frame["marker_size"]),
            c=node_frame["display_order"],
            cmap="Blues",
            edgecolors="white",
            linewidths=0.8,
            alpha=0.96,
            zorder=10,
            label="Nodes: color=time order, size=pupil, label=node index",
        )
        if show_node_index_labels:
            for row in node_frame.itertuples(index=False):
                ax.text(
                    row.x,
                    row.y,
                    row.node_label,
                    ha="center",
                    va="center",
                    fontsize=max(NODE_LABEL_FONT_SIZE - 2, 6),
                    color="#111111",
                    zorder=11,
                )
        ax.set_title(graph_name)
        ax.set_xlabel("x-avg")
        ax.set_ylabel("y-avg")
        ax.set_aspect("equal", adjustable="box")
        if invert_y_axis:
            ax.invert_yaxis()

    handles = list(relation_handles.values()) + [scatter]
    labels = [handle.get_label() for handle in handles]
    fig.legend(handles, labels, title="Edges and nodes", loc="center right")
    fig.colorbar(scatter, ax=axes, shrink=0.75, label="Node color: time order")
    fig.suptitle(
        f"subject={getattr(graph_v2, 'subject', None)} | recording={getattr(graph_v2, 'recording', None)} | "
        f"displayed nodes={len(node_indices)}/{graph_v2['node'].num_nodes}"
    )
    return fig

## Visualize one window

The default view draws the whole selected shorter visualization window. If you set `MAX_NODES_TO_DRAW`, the notebook displays a contiguous node range, not random nodes; edges are kept only when both endpoints are visible.

In [8]:
selected_graph_index = int(graph_index_widget.value) if graph_index_widget is not None else default_graph_index
selected_node_start = int(node_start_widget.value) if node_start_widget is not None else NODE_START
selected_max_nodes_raw = int(max_nodes_widget.value) if max_nodes_widget is not None else (-1 if MAX_NODES_TO_DRAW is None else int(MAX_NODES_TO_DRAW))
selected_max_nodes = None if selected_max_nodes_raw < 0 else selected_max_nodes_raw

# __getitem__ adds graph.idx, which is useful in titles and downstream batching diagnostics.
graph_v1 = dataset_v1[selected_graph_index]
graph_v2 = dataset_v2[selected_graph_index]
feature_columns = list(dataset_v2.feature_columns)

print(metadata.loc[metadata["graph_index"] == selected_graph_index].to_string(index=False))
print(f"Resolved feature columns: {feature_columns}")

node_indices = displayed_node_indices(graph_v2["node"].num_nodes, selected_node_start, selected_max_nodes)
display(edge_count_table({"GNN v1": graph_v1, "GNN v2": graph_v2}, node_indices, MAX_EDGES_PER_RELATION))

if PLOTLY_AVAILABLE:
    fig = plot_graph_pair_plotly(
        graph_v1,
        graph_v2,
        feature_columns=feature_columns,
        node_start=selected_node_start,
        max_nodes=selected_max_nodes,
        max_edges_per_relation=MAX_EDGES_PER_RELATION,
        invert_y_axis=INVERT_Y_AXIS,
        show_curved_edges=SHOW_CURVED_EDGES,
        show_node_index_labels=SHOW_NODE_INDEX_LABELS,
    )
    fig.show()
else:
    fig = plot_graph_pair_matplotlib(
        graph_v1,
        graph_v2,
        feature_columns=feature_columns,
        node_start=selected_node_start,
        max_nodes=selected_max_nodes,
        max_edges_per_relation=MAX_EDGES_PER_RELATION,
        invert_y_axis=INVERT_Y_AXIS,
        show_curved_edges=SHOW_CURVED_EDGES,
        show_node_index_labels=SHOW_NODE_INDEX_LABELS,
    )
    plt.show()

 graph_index subject     recording                            source_file  num_nodes target
           0      P5 newyork_f.avi subject_P5_recording_newyork_f.avi.csv        123  [6.0]
Resolved feature columns: ['x-avg', 'y-avg', 'pupil-size-left-avg', 'pupil-size-right-avg', 'time-window-normalized', 'distance-avg', 'fixation-duration']


,graph,relation,total_directed_edges,visible_directed_edges,displayed_edges,collapsed_for_plot
0,GNN v1,spatial,333,333,168,165
1,GNN v1,temporal,486,486,243,243
2,GNN v2,spatial,333,333,168,165
3,GNN v2,temporal_forward,243,243,243,0
4,GNN v2,temporal_backward,243,243,243,0
5,GNN v2,fixation,702,702,351,351


## Relation diagnostics

Use this table to check whether the visible graph has the expected v1/v2 relation split before exporting figures or screenshots.

In [9]:
diagnostics = edge_count_table(
    {"GNN v1": graph_v1, "GNN v2": graph_v2},
    displayed_node_indices(graph_v2["node"].num_nodes, selected_node_start, selected_max_nodes),
    max_edges=None,
)
display(diagnostics)

node_frame = graph_node_frame(graph_v2, feature_columns, displayed_node_indices(graph_v2["node"].num_nodes, selected_node_start, selected_max_nodes))
pupil_summary = node_frame["pupil_mean"].describe(percentiles=[0.05, 0.5, 0.95]).to_frame("displayed mean pupil")
distance_summary = node_frame["distance_cm"].describe(percentiles=[0.05, 0.5, 0.95]).to_frame("displayed distance cm")
display(pd.concat([pupil_summary, distance_summary], axis=1))

,graph,relation,total_directed_edges,visible_directed_edges,displayed_edges,collapsed_for_plot
0,GNN v1,spatial,333,333,168,165
1,GNN v1,temporal,486,486,243,243
2,GNN v2,spatial,333,333,168,165
3,GNN v2,temporal_forward,243,243,243,0
4,GNN v2,temporal_backward,243,243,243,0
5,GNN v2,fixation,702,702,351,351


,displayed mean pupil,displayed distance cm
count,123.000000,123.000000
mean,3.306454,59.444820
std,0.247075,0.034408
min,3.050611,59.288995
5%,3.079430,59.376443
50%,3.148652,59.445813
95%,3.710915,59.489128
max,3.732822,59.500427


## Notes

- Node geometry always comes from the model node features, not from a graph layout algorithm.
- By default, the notebook loads only a small reproducible subject-recording subset before graph construction.
- It then constructs shorter 2-second visualization windows with the same graph rules instead of hiding arbitrary nodes from a 10-second graph.
- If `MAX_NODES_TO_DRAW` is set, the displayed nodes are a contiguous local node range, not a random sample.
- Marker size is robustly scaled within the displayed nodes, so size comparisons are local to this view.
- Node labels show the local displayed node index, starting at 0; hover text still includes the eye-to-tracker distance in centimeters.
- Edges are curved by relation by default and drawn with `EDGE_ALPHA = 0.60`. This keeps multiple relation types between the same node pair visible while preserving exact node endpoints.
- For plotting only, `COLLAPSE_BIDIRECTIONAL_EDGES_FOR_PLOT = True` collapses exact/reverse duplicates within the same relation. The training graph remains directed and unchanged.